# 20th-Century Cookbook Corpus Slice (1910–1940)

Regenerate — **do not hand-edit** the `.ipynb` JSON:

```bash
uv run python scripts/generate_notebooks.py --name ldk2025_cookbook_slice
```

This notebook explores the 20th-century Dutch cookbook corpus (`LDK2025/corpus`):
1. **Corpus Ingest & KWIC Discovery:** Parse recipe sections (1910, 1912, 1925, 1940) and match food vocabulary terms.
2. **Slice Extraction:** Sample a focused KWIC input batch.
3. **Pipeline Annotation (A→B→C):** Run the LLM annotation pipeline with Structured Outputs.
4. **Qualia Analysis:** Inspect Step A entities, Step B macro-frames, and Step C preparation/preservation qualia.


## 1. Setup & Corpus Discovery

In [14]:
# put auto reload
%load_ext autoreload
%autoreload 2

In [1]:
from pathlib import Path
import re
import json
import pandas as pd
from tqdm.auto import tqdm

from data_io import resolve, load_jsonl, save_jsonl, save_parquet
from trifecta_annotation.vocabulary import resolve_thesaurus_lookup
from trifecta_annotation.schemas import KwicInput, TrifectaAnnotation
from trifecta_annotation.pipeline import annotate_record
from trifecta_annotation.analysis_export import annotations_to_analysis_frame

corpus_dir = Path(resolve('ldk2025_cookbooks'))
cookbook_files = sorted(corpus_dir.glob('*.txt'))
print(f'Found {len(cookbook_files)} cookbook files in {corpus_dir}:')
for f in cookbook_files:
    print(f'  - {f.name} ({f.stat().st_size / 1024:.1f} KB)')


Found 4 cookbook files in /Users/rikhoekstra/develop/LDK2025/corpus:
  - cookbook_1910.txt (517.8 KB)
  - cookbook_1912.txt (543.5 KB)
  - cookbook_1925.txt (541.5 KB)
  - cookbook_1940.txt (633.4 KB)


## 2. Recipe Extraction & KWIC Matcher

In [2]:
thesaurus_lookup = resolve_thesaurus_lookup()
print(f'Loaded food thesaurus with {len(thesaurus_lookup)} indexed surface terms.')

PRESERVATION_TERMS = {
    'inmaken', 'inmaak', 'wecken', 'pekelen', 'gepekelde', 'pekelvleesch', 'zouten', 'gezouten',
    'zoutevisch', 'rooken', 'gerookte', 'rookworst', 'droogen', 'drogen', 'gedroogde',
    'gelei', 'confituur', 'jam', 'azijn', 'inmaakazijn', 'steriliseeren', 'steriliseren',
}

def extract_recipe_snippets(
    file_path: Path,
    max_snippets_per_file: int = 50,
    include_preservation: bool = True,
) -> list[dict]:
    text = file_path.read_text(encoding='utf-8', errors='replace')
    year_match = re.search(r'\d{4}', file_path.name)
    year = year_match.group(0) if year_match else '1900'
    
    # Split by numbered recipes (e.g., '1. Radijsjes.', '2. Garnalen.')
    raw_recipes = re.split(r'\n\s*\d+\.\s+', text)
    general_snippets = []
    pres_snippets = []
    
    for idx, recipe in enumerate(raw_recipes[1:], start=1):
        cleaned = re.sub(r'\s+', ' ', recipe).strip()
        if len(cleaned) < 40:
            continue
        
        words = re.findall(r'\b[a-zA-ZÀ-ÿ\-]+\b', cleaned.lower())
        found_term = None
        for w in words:
            if len(w) > 2 and w in thesaurus_lookup:
                found_term = w
                break
        
        if not found_term:
            continue

        is_pres = any(pt in words or pt in cleaned.lower() for pt in PRESERVATION_TERMS)
        rec = {
            'record_id': f'ldk2025_{year}_rec{idx}_{found_term}',
            'corpus': 'ldk2025_cookbooks',
            'target_word': found_term,
            'context_text': cleaned[:350],
            'date': year,
            'source_path': file_path.name,
            'text_regime': 'RECIPE_PRACTICE',
            'is_preservation_candidate': is_pres,
        }
        
        if is_pres:
            pres_snippets.append(rec)
        else:
            general_snippets.append(rec)

    # Balance general preparation with explicit preservation recipes
    if include_preservation and pres_snippets:
        n_pres = min(len(pres_snippets), max(3, max_snippets_per_file // 2))
        n_gen = min(len(general_snippets), max_snippets_per_file - n_pres)
        return general_snippets[:n_gen] + pres_snippets[:n_pres]
    return general_snippets[:max_snippets_per_file]

all_candidates = []
for f in cookbook_files:
    extracted = extract_recipe_snippets(f, max_snippets_per_file=15, include_preservation=True)
    all_candidates.extend(extracted)

print(f'Extracted {len(all_candidates)} balanced candidate KWIC records across all 4 cookbooks.')
df_candidates = pd.DataFrame(all_candidates)
print(f"Preservation candidates in pool: {df_candidates['is_preservation_candidate'].sum()} / {len(df_candidates)}")
display(df_candidates.head(10))


Loaded food thesaurus with 1667 indexed surface terms.
Extracted 60 balanced candidate KWIC records across all 4 cookbooks.
Preservation candidates in pool: 28 / 60


,record_id,corpus,target_word,context_text,date,source_path,text_regime,is_preservation_candidate
0,ldk2025_1910_rec2_garnalen,ldk2025_cookbooks,garnalen,Garnalen met mayonaise. Behandel de garnalen z...,1910,cookbook_1910.txt,RECIPE_PRACTICE,False
1,ldk2025_1910_rec3_haring,ldk2025_cookbooks,haring,Haring met tomatenpurée. Week de haring eenige...,1910,cookbook_1910.txt,RECIPE_PRACTICE,False
2,ldk2025_1910_rec4_haring,ldk2025_cookbooks,haring,"Haring met mayonaise. Maak de haring schoon, z...",1910,cookbook_1910.txt,RECIPE_PRACTICE,False
3,ldk2025_1910_rec6_komkommer,ldk2025_cookbooks,komkommer,"Komkommer met room. Behandel de komkommer, zoo...",1910,cookbook_1910.txt,RECIPE_PRACTICE,False
4,ldk2025_1910_rec8_schil,ldk2025_cookbooks,schil,Selderijknol met mayonaise. Snijd den selderij...,1910,cookbook_1910.txt,RECIPE_PRACTICE,False
5,ldk2025_1910_rec9_ansjovis,ldk2025_cookbooks,ansjovis,Ansjovis. Week de ansjovissen eenige uren in w...,1910,cookbook_1910.txt,RECIPE_PRACTICE,False
6,ldk2025_1910_rec10_citroensap,ldk2025_cookbooks,citroensap,Ansjovissandwiches. 6 dunne sneedjes casinobro...,1910,cookbook_1910.txt,RECIPE_PRACTICE,False
7,ldk2025_1910_rec12_brood,ldk2025_cookbooks,brood,Geroosterd brood met sardines. 3 sneedjes oudb...,1910,cookbook_1910.txt,RECIPE_PRACTICE,False
8,ldk2025_1910_rec1_garnalen,ldk2025_cookbooks,garnalen,"Garnalen. Wasch de garnalen met zout water af,...",1910,cookbook_1910.txt,RECIPE_PRACTICE,True
9,ldk2025_1910_rec5_komkommer,ldk2025_cookbooks,komkommer,Komkommersla. Snijd de bittere punten van de k...,1910,cookbook_1910.txt,RECIPE_PRACTICE,True


## 3. Save Slice & Run Pipeline (A→B→C)

In [3]:
kwic_inputs_path = resolve('ldk2025_kwic_inputs')
save_jsonl(
    all_candidates,
    logical_name='ldk2025_kwic_inputs',
    script='ldk2025_cookbook_slice.ipynb',
)
print(f'Wrote KWIC inputs -> {kwic_inputs_path}')

# Run batch annotation (sample of 20 or full slice)
sample_to_annotate = all_candidates
annotations = []

print(f'Annotating sample of {len(sample_to_annotate)} records with Ollama / Qwen...')
for item in tqdm(sample_to_annotate):
    inp = KwicInput.model_validate(item)
    ann = annotate_record(inp)
    annotations.append(ann.model_dump(mode='json'))

annotations_path = resolve('ldk2025_annotations')
save_jsonl(
    annotations,
    logical_name='ldk2025_annotations',
    script='ldk2025_cookbook_slice.ipynb',
)
print(f'Saved annotations -> {annotations_path}')
print(f'Saved annotations -> {annotations_path}')


Wrote KWIC inputs -> /Volumes/Extreme SSD/scratch/trifecta/ldk2025_kwic_inputs.jsonl
Annotating sample of 60 records with Ollama / Qwen...


  0%|          | 0/60 [00:00<?, ?it/s]

Saved annotations -> /Volumes/Extreme SSD/scratch/trifecta/ldk2025_annotations.jsonl
Saved annotations -> /Volumes/Extreme SSD/scratch/trifecta/ldk2025_annotations.jsonl


## 4. Flatten & Qualia Analysis

In [4]:
df_analysis = annotations_to_analysis_frame(annotations)
print(f'Analysis table shape: {df_analysis.shape}')

# Macro-frame distribution
print('\n--- Step B Macro-Frame Counts ---')
print(df_analysis['selected_frame'].value_counts())

# Display Step C extracted fields
cols_to_show = [
    'target_word', 'date', 'selected_frame', 'lexical_unit',
    'COOKING_CREATION_Method', 'COOKING_CREATION_Process', 'COOKING_CREATION_Food_Product',
    'PR_Technique', 'PR_Medium', 'PR_Food_Patient',
]
display(df_analysis[cols_to_show].fillna(''))


Analysis table shape: (60, 36)

--- Step B Macro-Frame Counts ---
selected_frame
COOKING_CREATION    60
Name: count, dtype: int64


,target_word,date,selected_frame,lexical_unit,COOKING_CREATION_Method,COOKING_CREATION_Process,COOKING_CREATION_Food_Product,PR_Technique,PR_Medium,PR_Food_Patient
0,garnalen,1910,COOKING_CREATION,Behandel,Behandel de garnalen zooals in No. 2 aangegeve...,,Garnalen met mayonaise,,,
1,haring,1910,COOKING_CREATION,Week,Week de haring eenige uren in water met wat melk,"Snijd er den kop af, maak eene inkerving over ...",Haring met tomatenpurée,,,
2,haring,1910,COOKING_CREATION,Snijd ze in schuine mooten,Snijd ze in schuine mooten,,Haring met mayonaise,,,
3,komkommer,1910,COOKING_CREATION,giet,"Laat de reepjes, met zout bestrooid, eenigen t...",,Komkommer met room,,,
4,schil,1910,COOKING_CREATION,schil,schil deze,,,,,
5,ansjovis,1910,COOKING_CREATION,ansjovissen,Week de ansjovissen eenige uren in water met e...,,ansjovis,,,
6,citroensap,1910,COOKING_CREATION,week ze gedurende io minuten,week ze gedurende io minuten,,,,,
7,brood,1910,COOKING_CREATION,roosterd,Rooster ze op een bakblik lichtbruin inden oven,"sneedjes oudbakken casinobrood, snijd zeer dun...",Geroosterd brood met sardines,,,
8,garnalen,1910,COOKING_CREATION,Wasch,"Wasch de garnalen met zout water af, laat ze o...",,garnalen,,,
9,komkommer,1910,COOKING_CREATION,maak er met een sambalschaafje dunne reepjes van,maak er met een sambalschaafje dunne reepjes van,,reepjes van komkommer,,,


## 5. Summary & Observations

- **Frame distribution:** In 20th-century recipe practice, `COOKING_CREATION` dominates along with selective `PRESERVING`.
- **Step C Qualia clarity:** Modern 20th-century Dutch syntax leads to cleaner method and product spans.
- **Integration:** This corpus provides a natural modern bridge for comparative historical analyses against the VOC corpora.


## 5. Comparative Analysis: VOC Recipes vs 20th-Century Cookbooks

Compare macro-frames, culinary triggers, and preservation techniques across historical eras.

In [5]:
# Load historical VOC analysis dataset for side-by-side comparison
try:
    df_voc_analysis = pd.read_parquet(resolve('trifecta_analysis'))
    has_voc = True
    print(f"Loaded {len(df_voc_analysis)} historical analysis records from trifecta_analysis.")
except Exception as e:
    has_voc = False
    print(f"Could not load trifecta_analysis ({e}). Using slice data only.")

if has_voc:
    # 1. Macro-Frame Distribution Comparison
    print("=" * 60)
    print("1. MACRO-FRAME DISTRIBUTION (VOC vs 20th-C. Cookbooks)")
    print("=" * 60)
    voc_frames = df_voc_analysis['selected_frame'].value_counts(normalize=True).rename('VOC_share')
    ldk_frames = df_analysis['selected_frame'].value_counts(normalize=True).rename('20c_Cookbooks_share')
    frame_comp = pd.concat([voc_frames, ldk_frames], axis=1).fillna(0.0)
    display(frame_comp.map(lambda v: f"{v:.1%}"))

    # 2. Top Culinary Trigger Verbs (lexical_unit)
    print("\n" + "=" * 60)
    print("2. TOP TRIGGER VERBS (lexical_unit)")
    print("=" * 60)
    voc_lus = df_voc_analysis[df_voc_analysis['selected_frame'] == 'COOKING_CREATION']['lexical_unit'].str.lower().value_counts().head(10).rename('VOC_Cooking_LUs')
    ldk_lus = df_analysis[df_analysis['selected_frame'] == 'COOKING_CREATION']['lexical_unit'].str.lower().value_counts().head(10).rename('20c_Cooking_LUs')
    display(pd.concat([voc_lus.reset_index(), ldk_lus.reset_index()], axis=1).fillna(''))

    # 3. Preparation Processes (COOKING_CREATION_Process)
    print("\n" + "=" * 60)
    print("3. TOP PREPARATION PROCESSES (COOKING_CREATION_Process)")
    print("=" * 60)
    voc_proc = df_voc_analysis[df_voc_analysis['COOKING_CREATION_Process'] != '']['COOKING_CREATION_Process'].str.lower().value_counts().head(8).rename('VOC_Processes')
    ldk_proc = df_analysis[df_analysis['COOKING_CREATION_Process'] != '']['COOKING_CREATION_Process'].str.lower().value_counts().head(8).rename('20c_Processes')
    display(pd.concat([voc_proc.reset_index(), ldk_proc.reset_index()], axis=1).fillna(''))


Loaded 308 historical analysis records from trifecta_analysis.
1. MACRO-FRAME DISTRIBUTION (VOC vs 20th-C. Cookbooks)


,VOC_share,20c_Cookbooks_share
selected_frame,,
NONE,26.3%,0.0%
COOKING_CREATION,25.0%,100.0%
,18.2%,0.0%
CURE,13.0%,0.0%
INGESTION,11.4%,0.0%
PRESERVING,6.2%,0.0%



2. TOP TRIGGER VERBS (lexical_unit)


,lexical_unit,VOC_Cooking_LUs,lexical_unit,20c_Cooking_LUs
0,koken,6,behandel,5
1,kookt,5,snijd ze in schuine mooten,4
2,braden,4,giet,4
3,kopen,2,trekken,4
4,doet,2,week,3
5,geven,2,schil,3
6,gekookt,2,ansjovissen,3
7,kooktse,2,maak er met een sambalschaafje dunne reepjes van,3
8,braaden,2,week ze gedurende io minuten,2
9,verjuis,1,wasch,2



3. TOP PREPARATION PROCESSES (COOKING_CREATION_Process)


,COOKING_CREATION_Process,VOC_Processes,COOKING_CREATION_Process,20c_Processes
0,thermal process applied during cooking,2,"snijd er den kop af, maak eene inkerving over ...",3
1,soep koken,2,trekken,3
2,slachten,1,week gemaakt,2
3,opgesogt,1,"sneedjes oudbakken casinobrood, snijd zeer dun...",1
4,"ghelijck'tde la farine, tout ainsi vanden meul...",1,sneedjes,1
5,"lywaat lopen, zout smelten, honig en olyven-, ...",1,"gerookte zalm, mayonaise of remouladesaus, sni...",1
6,gerezen,1,"snijd den selderijknol in zeer dunne plakken, ...",1
7,stoving,1,sneid zeer dunne sneetjes casinobrood,1


In [6]:
# Annotate the remaining candidates from all 4 cookbooks
remaining_candidates = all_candidates[len(annotations):]

if remaining_candidates:
    print(f"Annotating remaining {len(remaining_candidates)} records with Ollama / Qwen...")
    for item in tqdm(remaining_candidates):
        inp = KwicInput.model_validate(item)
        ann = annotate_record(inp)
        annotations.append(ann.model_dump(mode='json'))

    # Update saved annotations
    annotations_path = resolve('ldk2025_annotations')
    save_jsonl(
        annotations,
        logical_name='ldk2025_annotations',
        script='ldk2025_cookbook_slice.ipynb',
    )
    print(f"Updated full annotations dataset ({len(annotations)} records) -> {annotations_path}")

    # Re-generate analysis frame
    df_analysis = annotations_to_analysis_frame(annotations)
    print(f"Updated analysis frame: {df_analysis.shape}")
    print("\nUpdated Step B Counts across all cookbooks:")
    print(df_analysis['selected_frame'].value_counts())
else:
    print("All candidate records are already annotated.")


All candidate records are already annotated.


In [7]:
import pandas as pd
from data_io import resolve

df_full = pd.read_parquet(resolve('trifecta_analysis').parent / 'ldk2025_analysis.parquet')
print(f"Loaded {len(df_full)} annotated 20th-century recipes.")

# Macro-frame breakdown
print(df_full['selected_frame'].value_counts())

# Top preparation methods across all 4 cookbooks
print(df_full[df_full['selected_frame'] == 'COOKING_CREATION']['COOKING_CREATION_Method'].value_counts().head(10))

Loaded 3663 annotated 20th-century recipes.
selected_frame
COOKING_CREATION    3601
                      43
PRESERVING            10
CURE                   7
NONE                   2
Name: count, dtype: int64
COOKING_CREATION_Method
                                     252
gaar koken                            90
koken                                 47
feuilletée                            46
meng                                  41
Bereid                                37
kook                                  35
kook ze                               34
Bereid de saus op de gewone wijze     31
gaar                                  31
Name: count, dtype: int64


In [9]:
print(
    df_full.loc[
        df_full['selected_frame'].eq('PRESERVING'),
        'PR_Technique',
    ].value_counts().head(10)
)

PR_Technique
droog worden                            4
pickling                                4
bewaren met zout, salpeter en suiker    1
bewaren in alcohol                      1
Name: count, dtype: int64


In [17]:
df_full.target_word.value_counts().head(10)

target_word
melk           331
boter          292
eieren         246
water          230
suiker         193
zout           161
bouillon       122
bloem          103
aardappelen     75
brood           74
Name: count, dtype: int64

In [16]:
# Force-reload data_io in running kernel and export to TSV/CSV
import importlib
import data_io
import data_io.manifest

importlib.reload(data_io.manifest)
importlib.reload(data_io)

data_io.manifest._default_manager = None
csv_path = data_io.resolve('ldk2025_analysis_csv')
df_full.to_csv(csv_path, index=False, sep='\t')
print(f"Saved analysis TSV/CSV -> {csv_path}")


Saved analysis TSV/CSV -> /Volumes/Extreme SSD/scratch/trifecta/analysis/ldk2025_analysis.csv


## 6. Comprehensive Corpus Statistics (Full 3,663 Recipes)

Overview of the entire 1910–1940 cookbook corpus:
1. **Corpus & Volume Summary:** Total recipes, words, target terms across eras.
2. **Macro-Frame Breakdown across Cookbook Years (1910, 1912, 1925, 1940).**
3. **Top Ingredients / Food Entities** with their dominant culinary roles.
4. **Qualia Action Landscape:** Most frequent cooking methods, processes, and resulting food products.
5. **Preservation Practices:** Breakdown of modern 20th-century preservation techniques and media.

In [18]:
import pandas as pd
import numpy as np
from data_io import resolve

df_full = pd.read_parquet(resolve('ldk2025_analysis'))

# 1. Corpus Volume & Timeline Breakdown
print("=" * 70)
print("1. CORPUS VOLUME & TIMELINE (1910–1940)")
print("=" * 70)

timeline = df_full.groupby('date').agg(
    recipes=('record_id', 'count'),
    unique_targets=('target_word', 'nunique'),
    cooking_creation=('selected_frame', lambda s: (s == 'COOKING_CREATION').sum()),
    preserving=('selected_frame', lambda s: (s == 'PRESERVING').sum()),
    cure=('selected_frame', lambda s: (s == 'CURE').sum()),
    dropped=('dropped', 'sum')
)
display(timeline)

# 2. Top 20 Food Targets & Their Dominant Cooking Triggers
print("\n" + "=" * 70)
print("2. TOP 20 INGREDIENTS & DOMINANT PREPARATION TRIGGERS")
print("=" * 70)

top_targets = df_full['target_word'].value_counts().head(20).index
target_stats = []
for target in top_targets:
    sub = df_full[df_full['target_word'] == target]
    top_verbs = sub['lexical_unit'].str.lower().value_counts().head(3).index.tolist()
    top_methods = sub[sub['COOKING_CREATION_Method'] != '']['COOKING_CREATION_Method'].str.lower().value_counts().head(2).index.tolist()
    target_stats.append({
        'target_word': target,
        'count': len(sub),
        'top_trigger_verbs': ", ".join(top_verbs),
        'top_methods': ", ".join(top_methods[:2])
    })
display(pd.DataFrame(target_stats))

# 3. Macro-Frame Distribution across Eras (Crosstab)
print("\n" + "=" * 70)
print("3. MACRO-FRAME DISTRIBUTION PER YEAR")
print("=" * 70)
frame_xtab = pd.crosstab(df_full['date'], df_full['selected_frame'], margins=True)
display(frame_xtab)

# 4. Qualia Extraction Coverage
print("\n" + "=" * 70)
print("4. STEP C QUALIA EXTRACTION FILL RATES")
print("=" * 70)
qualia_cols = [
    'COOKING_CREATION_Method', 'COOKING_CREATION_Process', 'COOKING_CREATION_Food_Product',
    'PR_Technique', 'PR_Medium', 'PR_Food_Patient',
    'CURE_Affliction', 'CURE_Food_Treatment',
    'INGESTION_Context', 'INGESTION_Manner'
]
fill_rates = []
for col in qualia_cols:
    non_empty = (df_full[col].astype(str).str.strip() != '').sum()
    fill_rates.append({
        'field': col,
        'filled_count': non_empty,
        'fill_rate': f"{non_empty / len(df_full):.1%}"
    })
display(pd.DataFrame(fill_rates))

# 5. Top 15 Resulting Food Products (COOKING_CREATION_Food_Product)
print("\n" + "=" * 70)
print("5. TOP 15 RESULTING DISHES & PRODUCTS (COOKING_CREATION_Food_Product)")
print("=" * 70)
top_products = df_full[df_full['COOKING_CREATION_Food_Product'] != '']['COOKING_CREATION_Food_Product'].value_counts().head(15)
display(top_products.to_frame(name='count'))


1. CORPUS VOLUME & TIMELINE (1910–1940)


,recipes,unique_targets,cooking_creation,preserving,cure,dropped
date,,,,,,
1910,875,144,861,2,1,11
1912,890,145,871,2,2,13
1925,868,145,851,3,3,11
1940,1030,153,1018,3,1,8



2. TOP 20 INGREDIENTS & DOMINANT PREPARATION TRIGGERS


,target_word,count,top_trigger_verbs,top_methods
0,melk,331,"aan de kook, kook, bereid","gaar koken, bereid"
1,boter,292,"bereiding, bereid op de gewone wijze, roer","feuilletée, roer de boter tot room"
2,eieren,246,"klop, bereid, roer","gereid, klop"
3,water,230,"kook, bereid, kokend water","giet, gaar koken"
4,suiker,193,"bereid, inkoken, bereiding","inkoken, meng in"
5,zout,161,"zout, kook ze, bereid ze als bruine boonen","zout, roerende"
6,bouillon,122,"bouillon, bereid, kook","bereid de saus op de gewone wijze, trek op de ..."
7,bloem,103,"kneed, bereid, roer","rijzen, kneed alle ingrediënten tot een stevig..."
8,aardappelen,75,"koken, bakken, gebakken","zet ze op, oude aardappelen koken"
9,brood,74,"week, bak, wrijf","bak, wrijven"



3. MACRO-FRAME DISTRIBUTION PER YEAR


selected_frame,,COOKING_CREATION,CURE,NONE,PRESERVING,All
date,,,,,,
1910,11,861,1,0,2,875
1912,13,871,2,2,2,890
1925,11,851,3,0,3,868
1940,8,1018,1,0,3,1030
All,43,3601,7,2,10,3663



4. STEP C QUALIA EXTRACTION FILL RATES


,field,filled_count,fill_rate
0,COOKING_CREATION_Method,3349,91.4%
1,COOKING_CREATION_Process,2073,56.6%
2,COOKING_CREATION_Food_Product,2783,76.0%
3,PR_Technique,10,0.3%
4,PR_Medium,6,0.2%
5,PR_Food_Patient,10,0.3%
6,CURE_Affliction,6,0.2%
7,CURE_Food_Treatment,7,0.2%
8,INGESTION_Context,0,0.0%
9,INGESTION_Manner,0,0.0%



5. TOP 15 RESULTING DISHES & PRODUCTS (COOKING_CREATION_Food_Product)


,count
COOKING_CREATION_Food_Product,
bouillon,49
eieren,36
rijst,31
saus,28
soep,27
bruine boonen,26
taart,24
deeg,22
visch,22
